In [22]:
import pandas as pd
import altair as alt
import panel as pn
from panel.interact import interact
from vega_datasets import data

pn.extension('vega')
alt.renderers.enable("browser")
imd_df = pd.read_csv('country_economics_data.csv')
interval = alt.selection_interval()
region_options = sorted(imd_df["Region"].unique())
selection = alt.selection_point(fields=['Region'], name='Region')
region_dropdown = alt.binding_select(options=[None] + region_options, name="Region")
region_select = alt.selection_point(fields=['Region'], bind=region_dropdown)

country_select = alt.selection_single(
    fields=['Name'],
    on='click',
    clear='dblclick'
)


def region(base):
    filter_region = base.add_params(
        region_select
    ).transform_filter(
        region_select
    )
    return filter_region

C:\Users\Clo\AppData\Local\Temp\ipykernel_30552\643643267.py:7: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension('vega')


C:\Users\Clo\AppData\Local\Temp\ipykernel_30552\643643267.py:16: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  country_select = alt.selection_single(


In [23]:
def top_gdp():
    chart = (
        alt.Chart(imd_df)
        # 1️⃣ Filter data by region (from your selection)
        .transform_filter(selection)
        # 2️⃣ Rank GDP *within* the filtered dataset
        .transform_window(
            rank='rank(GDP)',
            sort=[alt.SortField('GDP', order='descending')],
            groupby=['Region']  # 🧠 ensures ranking restarts per region
        )
        # 3️⃣ Keep only top 10 per region
        .transform_filter('datum.rank <= 10')
        .mark_bar()
        .encode(
            x=alt.X('GDP:Q', title='GDP (in billions)'),
            y=alt.Y('Name:N', sort='-x', title='Country'),
            color=alt.Color('Region:N', legend=None),
        )
        .properties(
            title="Top GDPs (updates by region)",
            width=375,
            height=545
        ))
    return chart


In [24]:
def world_map():
    # Clean up the country names
    imd_df['Name'] = imd_df['Name'].str.strip()

    # Load world geometry
    countries = alt.topo_feature(data.world_110m.url, 'countries')

    # Variable to color by
    variable = 'Population'

    # Create the chart
    world_map = alt.Chart(countries).mark_geoshape(
        stroke='white',
        strokeWidth=0.5
    ).encode(
        color=alt.Color(
            f"{variable}:Q",
            title=f"{variable} (Millions)",
            legend=alt.Legend(
                orient="bottom",
                gradientLength=960,
                gradientThickness=10
            )
        ),
        tooltip=[
            alt.Tooltip('Name:N', title='Country'),
            alt.Tooltip('Population:Q', title='Population (Millions)'),
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('Capital:N', title='Capital')
        ]
    ).transform_lookup(
        lookup='id',
        from_=alt.LookupData(imd_df, 'ID', ['Name', 'GDP', 'Population', 'Region', 'Capital'])
    ).project(
        type='mercator'
    ).properties(
        width=900,
        height=545,
        title='Population by Country'
    ).add_selection(
    country_select)

    return world_map


In [25]:
'''
import altair as alt

# define global selection once
interval = alt.selection_interval()

def gdp_popu():
    return (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('GDP:Q', title='GDP (in billions)'),
            y=alt.Y('Population:Q', title='Population (millions)'),
            color=alt.Color('Region:N'),
            tooltip=[
            alt.Tooltip('Name:N', title='Country'),
            alt.Tooltip('Population:Q', title='Population (Millions)'),
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('GDP:Q', title='GDP (in billions)')
        ]
        )
        .transform_filter(selection)
        .properties(title="GDP vs Population",
                    width = 550,
                    height = 270)
    ).transform_filter(country_select)

def int_inf():
    return (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('Interest Rate:Q', title='Interest'),
            y=alt.Y('Inflation Rate:Q', title='Inflation'),
            color=alt.Color('Region:N'),
            tooltip=[
            alt.Tooltip('Name:N', title='Country'),
            alt.Tooltip('Inflation Rate:Q', title='Inflation'),
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('Interest Rate:Q', title='Interest')
        ]
        )
        .transform_filter(selection)
        .properties(title="Interest vs Inflation",
                    width = 550,
                    height = 270)
    ).transform_filter(country_select)

def GDP_grow_inf():
    return (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('Inflation Rate:Q', title='GDP Growth'),
            y=alt.Y('GDP Growth:Q', title='Inflation'),
            color=alt.Color('Region:N', legend=alt.Legend(orient="bottom")),
            tooltip=[
            alt.Tooltip('Name:N', title='Country'),
            alt.Tooltip('Inflation Rate:Q', title='Inflation'),
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('GDP Growth:Q', title='GDP Growth')
        ]
        )
        .transform_filter(selection)
        .properties(title="GDP Growth vs Inflation",
                    width = 550,
                    height = 270)
    ).transform_filter(country_select)

def line_of_bf(base, x, y): 
    regression_line = ( 
        alt.Chart(imd_df)
        .transform_regression(x, y) 
        .mark_line(color='red') 
        .encode( 
            x=x, 
            y=y 
            ) ).transform_filter(country_select) 
    return base + regression_line
'''
import altair as alt

# Define global selections
highlight = alt.selection_point(on='click', fields=['Name'], nearest=True)
zoom = alt.selection_interval(bind='scales')  # enables zoom & pan

def gdp_popu():
    base = (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('GDP:Q', title='GDP (in billions)'),
            y=alt.Y('Population:Q', title='Population (millions)'),
            color=alt.Color('Region:N'),
            shape=alt.Color('Region:N'),
            size=alt.condition(highlight, alt.value(150), alt.value(60)),
            opacity=alt.condition(highlight, alt.value(1), alt.value(0.3)),
            tooltip=[
                alt.Tooltip('Name:N', title='Country'),
                alt.Tooltip('Population:Q', title='Population (Millions)'),
                alt.Tooltip('Region:N', title='Region'),
                alt.Tooltip('GDP:Q', title='GDP (in billions)')
            ]
        )
        .add_params(highlight, zoom)
        .transform_filter(country_select)
        .properties(title="GDP vs Population", width=550, height=270)
    )
    return base

def int_inf():
    base = (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('Interest Rate:Q', title='Interest'),
            y=alt.Y('Inflation Rate:Q', title='Inflation'),
            color=alt.Color('Region:N'),
            shape=alt.Color('Region:N'),
            size=alt.condition(highlight, alt.value(150), alt.value(60)),
            opacity=alt.condition(highlight, alt.value(1), alt.value(0.3)),
            tooltip=[
                alt.Tooltip('Name:N', title='Country'),
                alt.Tooltip('Inflation Rate:Q', title='Inflation'),
                alt.Tooltip('Region:N', title='Region'),
                alt.Tooltip('Interest Rate:Q', title='Interest')
            ]
        )
        .add_params(highlight, zoom)
        .transform_filter(country_select)
        .properties(title="Interest vs Inflation", width=550, height=270)
    )
    return base

def GDP_grow_inf():
    base = (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('Inflation Rate:Q', title='Inflation'),
            y=alt.Y('GDP Growth:Q', title='GDP Growth'),
            color=alt.Color('Region:N', legend=alt.Legend(orient="bottom")),
            shape=alt.Color('Region:N'),
            size=alt.condition(highlight, alt.value(150), alt.value(60)),
            opacity=alt.condition(highlight, alt.value(1), alt.value(0.3)),
            tooltip=[
                alt.Tooltip('Name:N', title='Country'),
                alt.Tooltip('Inflation Rate:Q', title='Inflation'),
                alt.Tooltip('Region:N', title='Region'),
                alt.Tooltip('GDP Growth:Q', title='GDP Growth')
            ]
        )
        .add_params(highlight, zoom)
        .transform_filter(country_select)
        .properties(title="GDP Growth vs Inflation", width=550, height=270)
    )
    return base

def line_of_bf(base, x, y):
    regression_line = (
        alt.Chart(imd_df)
        .transform_regression(x, y)
        .mark_line(color='red')
        .encode(x=x, y=y)
        .transform_filter(country_select)
    )
    return base + regression_line



In [26]:
def unemployment_region():
    return (
        alt.Chart(imd_df)
        .mark_boxplot(extent='min-max', color="#4C78A8")
        .encode(
            x=alt.X("Region:N", title="Region", sort="-y"),
            y=alt.Y("Jobless Rate:Q", title="Jobless Rate (%)"),
            tooltip=["Region", "Jobless Rate"],
            color=alt.Color('Region:N', legend=None),
        )
        .properties(
            title="Unemployment Rate by Region",
            width=375,
            height=545,
        )
    ).transform_filter(selection)


In [27]:
import altair as alt

imd_df["Fiscal Status"] = imd_df["Gov. Budget"].apply(lambda x: "Surplus" if x > 0 else "Deficit")

chart = (
    alt.Chart(imd_df)
    .mark_circle(size=90, opacity=0.75)
    .encode(
        x=alt.X("Debt/GDP:Q", title="Debt to GDP (%)"),
        y=alt.Y("Gov. Budget:Q", title="Budget Balance (% of GDP)"),
        color=alt.Color("Fiscal Status:N", scale=alt.Scale(domain=["Surplus", "Deficit"], range=["#2ca02c", "#d62728"])),
        shape=alt.Shape("Region:N", title="Region"),
        tooltip=["Region:N", "Debt/GDP:Q", "Gov. Budget:Q"]
    )
    .properties(
        title="Fiscal Balance vs Debt Load",
        width=700,
        height=450
    )
)


In [28]:
world_by_pop = region(world_map().add_params(selection))
top_10_gdps = region(top_gdp())
unemployment_region = unemployment_region()

chart1 = line_of_bf(int_inf(), 'Interest Rate', 'Inflation Rate')
chart2 = line_of_bf(gdp_popu(), 'GDP', 'Population')
chart3 = line_of_bf(GDP_grow_inf(), 'Inflation Rate', 'GDP Growth')

horizontal = alt.hconcat(top_10_gdps, world_by_pop, unemployment_region)
horizontal2 = region(alt.hconcat(chart1, chart2, chart3))

final_output = alt.vconcat(horizontal, horizontal2).properties(
    title=alt.TitleParams(
        text="Economic Dashboard Overview",
        subtitle="  ",
        fontSize=20,
        font='Arial',
        anchor='middle',
        color='darkblue'
    )
)
final_output


C:\Users\Clo\AppData\Local\Temp\ipykernel_30552\3896040767.py:40: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.VConcatChart(...)